# Applying `RCTD` and `MCube` to the 10x Visium CRC dataset

In [1]:
set.seed(20250502)

library(Matrix)
library(ggplot2)

library(spacexr)
library(MCube)

In [2]:
RAW_DATA_PATH <- "/import/home/share/zw/data/CRC"
DATA_PATH <- "/import/home/share/zw/pql/data/CRC"
RESULT_PATH <- "/import/home/share/zw/pql/results/CRC/Visium"

if (!dir.exists(file.path(RESULT_PATH))) {
    dir.create(file.path(RESULT_PATH), recursive = TRUE)
}

## Cell type deconvolution using `RCTD`

In [3]:
library(Seurat)

FlexRef <- Read10X_h5(file.path(
    RAW_DATA_PATH, "sc",
    "HumanColonCancer_Flex_Multiplex_count_filtered_feature_bc_matrix.h5"
))
# MetaData <- readRDS(file.path(
#     RAW_DATA_PATH, "sc", "FlexSeuratV5_MetaData.rds"
# )) # See FlexSingleCell.R if not generated.

meta <- read.csv(file.path(
    RAW_DATA_PATH, "HumanColonCancer_VisiumHD/MetaData/SingleCell_MetaData.csv.gz"
))

KpIdents <- names(which(table(meta$Level2) > 25))
meta <- meta[meta$Level2 %in% KpIdents, ]
FlexRef <- FlexRef[, meta$Barcode]

CTRef <- meta$Level2
CTRef <- gsub("/", "_", CTRef)
CTRef <- as.factor(CTRef)
names(CTRef) <- meta$Barcode

reference <- Reference(FlexRef, CTRef, colSums(FlexRef))

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Warning message in check_UMI(nUMI, "Reference", require_2d = T, require_int = require_int, :
“Reference: some nUMI values are less than min_UMI = 100, and these cells will be removed. Optionally, you may lower the min_UMI parameter.”
Warning message in Reference(FlexRef, CTRef, colSums(FlexRef)):
“Reference: number of cells per cell type is 34844, larger than maximum allowable of 10000. Downsampling number of cells to: 10000”


In [4]:
counts <- Read10X_h5(file.path(
    RAW_DATA_PATH, "visium",
    "Visium_V2_Human_Colon_Cancer_P2_filtered_feature_bc_matrix.h5"
))

coordinates <- as.data.frame(readr::read_csv(file.path(
    RAW_DATA_PATH, "visium",
    "spatial/tissue_positions.csv"
)))
rownames(coordinates)<-coordinates$barcode
coordinates<-coordinates[colnames(counts), ]
coordinates <- coordinates[, c(6, 5)]

puck <- SpatialRNA(coordinates, counts, colSums(counts))

Rows: 14336 Columns: 6
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): barcode
dbl (5): in_tissue, array_row, array_col, pxl_row_in_fullres, pxl_col_in_ful...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
myRCTD <- create.RCTD(puck, reference, max_cores = 12)
myRCTD <- run.RCTD(myRCTD, doublet_mode = "full")
saveRDS(myRCTD, file.path(RESULT_PATH, "myRCTD.rds"))
# myRCTD <- readRDS(file.path(RESULT_PATH, "myRCTD.rds"))

Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 186667

process_cell_type_info: number of genes in reference: 18082




              Adipocyte                     CAF              CD4 T cell 
                     83                   10000                   10000 
   CD8 Cytotoxic T cell             Endothelial           Enteric Glial 
                  10000                    6940                    3680 
             Enterocyte              Epithelial              Fibroblast 
                   1739                    7432                   10000 
                 Goblet   Lymphatic Endothelial              Macrophage 
                  10000                    1095                   10000 
                   Mast                Mature B                  mRegDC 
                   1925                    8318                     404 
          Myofibroblast          Neuroendocrine              Neutrophil 
                    972                     529                    3138 
                    pDC               Pericytes                  Plasma 
                    119                    3046   

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.3 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.3 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.3 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.0 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.3 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.3 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.3 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.1 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.3 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.3 GiB”
Warning message in asMethod(ob

## Cell-type-specific SVG identification using `MCube`

In [6]:
myRCTD <- readRDS(file.path(RESULT_PATH, "myRCTD.rds"))
spot_names <- colnames(myRCTD@spatialRNA@counts)
weights_RCTD <- as.matrix(myRCTD@results$weights)
proportions_RCTD <- weights_RCTD / rowSums(weights_RCTD)
spot_effects_RCTD <- log(rowSums(weights_RCTD))
names(spot_effects_RCTD) <- rownames(weights_RCTD)

In [7]:
mcube_object <- createMCube(
    counts = t(as.matrix(myRCTD@originalSpatialRNA@counts[, spot_names])),
    coordinates = as.matrix(myRCTD@spatialRNA@coords),
    proportions = weights_RCTD / rowSums(weights_RCTD),
    library_sizes = myRCTD@spatialRNA@nUMI,
    reference = t(myRCTD@cell_type_info$info[[1]]),
    used_for_deconvolution = rownames(myRCTD@spatialRNA@counts),
    spot_effects = spot_effects_RCTD,
    celltype_threshold = 50
)
mcube_object <- mcubeFitNull(
    mcube_object,
    num_workers = 70, num_threads = 1
)
mcube_object <- mcubeTest(
    mcube_object,
    num_workers = 70, num_threads = 1, shared_memory = TRUE
)

saveRDS(
    mcube_object,
    file = file.path(
        RESULT_PATH, 
        paste0("mcube", ".rds")
    )
)

The batch_id is not provided!
All spots are assumed to be from the same batch and share the same gene platform effects.

Select high-abundance cell types to analyze with proportion_threshold = 0.1 and celltype_threshold = 50.

mcubeFilterCellTypes: Cell type(s) CAF, Endothelial, Goblet, Macrophage, Plasma, Tumor III, vSM pass the threshold.

Cell type(s) Adipocyte, CD4 T cell, CD8 Cytotoxic T cell, Enteric Glial, Enterocyte, Epithelial, Fibroblast, Lymphatic Endothelial, Mast, Mature B, mRegDC, Myofibroblast, Neuroendocrine, Neutrophil, pDC, Pericytes, Proliferating Immune II, QC_Filtered, SM Stress Response, Smooth Muscle, Tuft, Tumor I, Tumor II, Tumor V, Unknown III (SM) has/have less than the minimum celltype_threshold = 50 with proportion_threshold = 0.1.
To include the above cell type(s), please reduce celltype_threshold or proportion_threshold.

Cell type(s) CAF, Endothelial, Goblet, Macrophage, Plasma, Tumor III, vSM will be analyzed.

Filter out lowly-expressed genes with gene